# Day 2 - Talking to an LLM, and measuring it honestly

**Big idea:** calling a model is easy. Knowing how fast and how expensive it *really* is takes a little care.

## 1. What an LLM API call actually is

You send a **prompt**, you get back **text** plus **usage** (how many tokens went in and came out). That's it. Gemini, OpenRouter, a local Ollama - they all follow this same shape; only the URL and field names change.

On Day 2 I sent the same prompt to three providers:

| Provider | Model | Median latency | Cost |
|---|---|---|---|
| Gemini | gemini-3.6-flash | 4.39 s | $0.000148 |
| OpenRouter | deepseek v4 flash (free) | 2.39 s | $0 |
| Ollama (local) | llama3.2:3b | 1.69 s | $0 |

## 2. Measuring time: `time.perf_counter()`

`perf_counter()` is a stopwatch. Read it before, read it after, subtract.

In [1]:
import time

start = time.perf_counter()
time.sleep(0.25)          # pretend this is a network call
elapsed = time.perf_counter() - start
print(f"took {elapsed:.3f} s")

took 0.255 s


## 3. Why the median of 3 (not one run, not the average)

One run can be unlucky: a cold connection, a busy server, a hiccup. The **average** gets dragged by that one bad run. The **median** (the middle value after sorting) just ignores it.

In [2]:
import statistics

runs = [1.21, 1.34, 9.80]   # two normal runs, one freak slow one
print("average:", round(statistics.mean(runs), 2), "s   <- dragged up by the one bad run")
print("median: ", round(statistics.median(runs), 2), "s   <- the 'typical' run")

average: 4.12 s   <- dragged up by the one bad run
median:  1.34 s   <- the 'typical' run


## 4. Cost math

Providers price **per 1 million tokens**, and input and output have **different prices**:

`cost = input_tokens / 1M x input_price  +  output_tokens / 1M x output_price`

In [3]:
def cost_usd(input_tokens, output_tokens, input_price_per_1m, output_price_per_1m):
    return (input_tokens / 1_000_000) * input_price_per_1m + (output_tokens / 1_000_000) * output_price_per_1m

# Recall-quiz example: $1 in / $5 out, 500 in / 200 out tokens
print(f"${cost_usd(500, 200, 1.00, 5.00):.4f}")   # 0.0005 + 0.0010 = 0.0015

$0.0015


### Same tokens, different bill

Two providers can use the same number of tokens and still charge very different amounts - the **rate** differs. Token count and price are separate things:

In [4]:
for name, price_in, price_out in [("Provider A", 1.00, 5.00), ("Provider B", 5.00, 25.00)]:
    print(f"{name}: ${cost_usd(1000, 1000, price_in, price_out):.4f} for 1,000 in + 1,000 out tokens")

Provider A: $0.0060 for 1,000 in + 1,000 out tokens
Provider B: $0.0300 for 1,000 in + 1,000 out tokens


## 5. `max_tokens` is a ceiling, not a purchase

Setting `max_tokens=4000` for a 20-token answer does **not** bill you for 4,000. You pay for what's actually generated. It's a safety rail: a lower limit just stops a runaway answer sooner.

## 6. "Ollama was fastest" doesn't mean "Ollama's model is fastest"

I measured **end-to-end** time. Ollama skips the internet entirely (it's on my laptop), and it's a tiny 3B model vs big cloud models on different hardware. Not a fair race. To compare raw model speed you'd remove network time and match hardware and model size.

## 7. (optional, live) Call my local Ollama and time it

In [5]:
import requests

def call_ollama(prompt, model="llama3.2:3b"):
    try:
        start = time.perf_counter()
        r = requests.post(
            "http://localhost:11434/api/generate",
            json={"model": model, "prompt": prompt, "stream": False},
            timeout=60,
        )
        r.raise_for_status()
        data = r.json()
        print(f"latency: {time.perf_counter() - start:.2f} s")
        print(f"tokens:  {data.get('prompt_eval_count')} in, {data.get('eval_count')} out")
        print(f"answer:  {data.get('response', '').strip()}")
    except requests.RequestException as e:
        print("Skipped - Ollama isn't reachable. Start it with `ollama serve`.", type(e).__name__)

call_ollama("In one sentence, what is a vector database?")

latency: 2.08 s
tokens:  35 in, 52 out
answer:  A vector database is a type of database that stores and indexes large amounts of data in a numerical vector space, allowing for efficient querying and retrieval of data based on similarities in feature vectors, often used in applications such as search, recommendation, and similarity search.


## Recap

- An LLM call = prompt in -> text + token usage out.
- Time with `perf_counter`, run 3x, report the **median**.
- `cost = tokens_in x price_in + tokens_out x price_out` (per 1M tokens).
- `max_tokens` caps output; you pay only for what's generated.
- End-to-end latency is not the same as model speed.